# Day 32: 3D Gaussian Splatting

### Outline for today:

- SfM output representation
  - Camera poses (useful)
  - 3D point cloud (not that useful)
- Things you might want to do:
  - Build a 3D model (meshes)
  - Measure the scene (meshes seem nice?)
  - Render novel views (could do this from meshes, or not)
- 3D Gaussian Splatting: ML models to render new views

### Learning Goals:

After class today, students will be able to:

- Define the novel-view rendering problem
- Explain how 3DGS uses gradient descent to optimize a set of Gaussians to render novel views
- Describe how a set of 2D Gaussians can be composited into an image

In [ ]:
# imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
%matplotlib inline

## Warmup: Gaussian distributions in various dimensions

First up: 1D

In [ ]:
def plot_1d_gaussian(mu, sigma, axes, **kwargs):
    # Generate data
    x = np.linspace(-6, 6, 100)
    y = norm.pdf(x, mu, sigma)
    axes.plot(x, y, label=f'μ={mu}, σ={sigma}', **kwargs)

# Plot
plt.figure(figsize=(10, 4))
ax = plt.gca()

plot_1d_gaussian(0, 1, ax, linewidth=4)
plot_1d_gaussian(-2, 1, ax)
plot_1d_gaussian(1, 2, ax)
plot_1d_gaussian(4, 1/2, ax)

plt.title('1D Gaussian Distribution')
plt.xlabel('x')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True)
plt.show()

So what are Gaussian distributions in 2D?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# Parameters
mu = [0, 0]  # Mean
cov = [[1, 0], [0, 1]]  # Covariance matrix

# Generate data
x, y = np.random.multivariate_normal(mu, cov, 5000).T

# Create grid for the probability density function
x_grid, y_grid = np.mgrid[-4:4:.02, -4:4:.02]
pos = np.empty(x_grid.shape + (2,))
pos[:, :, 0] = x_grid
pos[:, :, 1] = y_grid
rv = multivariate_normal(mu, cov)

# Plotting
fig = plt.figure(figsize=(15, 5))

# Scatter plot with ellipse
ax1 = fig.add_subplot(131)
ax1.scatter(x, y, s=1, alpha=0.5)
circle1 = plt.Circle(mu, 3, color='r', fill=False, linestyle='--')
ax1.add_patch(circle1)
ax1.set_xlim(-4, 4)
ax1.set_ylim(-4, 4)
ax1.set_title('Samples from a 2D Gaussian')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.grid(True)

# Color-shaded plot
ax2 = fig.add_subplot(132)
ax2.contourf(x_grid, y_grid, rv.pdf(pos), cmap='cividis')
circle2 = plt.Circle(mu, 3, color='r', fill=False, linestyle='--')
ax2.add_patch(circle2)
ax2.set_xlim(-4, 4)
ax2.set_ylim(-4, 4)
ax2.set_title('2D Gaussian: Probability Density')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.grid(True)

# 3D Surface plot
ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(x_grid, y_grid, rv.pdf(pos), cmap='cividis', edgecolor='none')
ax3.set_xlim(-4, 4)
ax3.set_ylim(-4, 4)
ax3.set_title('3D Plot: Density of a 2D Gaussian')
ax3.set_xlabel('X axis')
ax3.set_ylabel('Y axis')
ax3.set_zlabel('Probability Density')

# Draw a dashed circle on the 3D plot
ax3.plot(np.cos(np.linspace(0, 2*np.pi, 100)) * 3, 
         np.sin(np.linspace(0, 2*np.pi, 100)) * 3, 
         0, 'r--')

# Show the plots
plt.tight_layout()
plt.show()

#### Formulas:

The 1D Gaussian distribution is defined by this equation:

$$ f(x) = \frac{1}{\sqrt{2 \pi \sigma^2}} e^{-\frac{(x - \mu)^2}{2 \sigma^2}} $$

The part out front is simply a normalizing constant. There's no $x$ in it. If we left this term off, this function would still look like a Gaussian. But the area under the curve would be $ \sqrt{2 \pi \sigma^2} $. Since the area under a probability distribution should be 1, we divide by that amount.

The rest of the formula has these elements:

- A difference between $x$ and the center, $mu$
  - squared
- Dividing by the standard deviation (width)

In higher dimensions, $x$ and $mu$ become vectors. $p(x)$ is still a real number, because it still assigns a *probability density* to each $x$. The spread parameter $\sigma$ is subtler. It becomes a *covariance matrix* $\Sigma$. 

$$ 
f(\mathbf{x}) = \frac{1}{\sqrt{(2 \pi)^k |\mathbf{\Sigma}|}} \exp\left(-\frac{1}{2} (\mathbf{x} - \mathbf{\mu})^T \mathbf{\Sigma}^{-1} (\mathbf{x} - \mathbf{\mu})\right)
$$

**Don't skip!**: Run the plots above for a couple of $(\mu, \Sigma)$ combinations. Be sure to see what the covariance terms do.

**Q:** Why can't the spread of a 2D Gaussian be described by a 2D number anymore?

#### Let's draw a 3D Gaussian

The easiest way to do this is as a point cloud. But remember that we're looking at a function from $\mathbf{R}^3$ to $\mathbf{R}^1$, which has the property that the area under the curve is 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# Parameters
nsamples = 5000
mu = np.array([0, 0, 0])
cov = np.array(
    [[1, 0, 0], 
     [0, 0.1, -0.3], 
     [0, -0.3, 1]]
)

# Generate sample data
samples = np.random.multivariate_normal(mu, cov, nsamples)

# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(9, 9))
ax3d = fig.add_subplot(221, projection='3d')
alpha = 0.2

# 3D Scatter Plot
ax3d.scatter(samples[:, 0], samples[:, 1], samples[:, 2], s=10, alpha=alpha)
ax3d.quiver(0, 0, 0, 3, 0, 0, color='r', linewidth=2)
ax3d.quiver(0, 0, 0, 0, 3, 0, color='g', linewidth=2)
ax3d.quiver(0, 0, 0, 0, 0, 3, color='b', linewidth=2)
ax3d.text(3.5, 0, 0, 'X', color='r', fontsize=12, weight='bold')
ax3d.text(0, 3.5, 0, 'Y', color='g', fontsize=12, weight='bold')
ax3d.text(0, 0, 3.5, 'Z', color='b', fontsize=12, weight='bold')
ax3d.set_title('3D Gaussian Distribution')
ax3d.set_xlim(-4, 4)
ax3d.set_ylim(-4, 4)
ax3d.set_zlim(-4, 4)

# 2D XY Projection
axs[0, 1].scatter(samples[:, 0], samples[:, 1], s=10, alpha=alpha)
axs[0, 1].arrow(0, 0, 3, 0, color='r', head_width=0.2, linewidth=2)
axs[0, 1].arrow(0, 0, 0, 3, color='g', head_width=0.2, linewidth=2)
axs[0, 1].text(3.5, 0, 'X', color='r', fontsize=12, weight='bold')
axs[0, 1].text(0, 3.5, 'Y', color='g', fontsize=12, weight='bold')
axs[0, 1].set_title('XY Projection')
axs[0, 1].set_xlim(-4, 4)
axs[0, 1].set_ylim(-4, 4)
axs[0, 1].set_aspect('equal')

# 2D XZ Projection
axs[1, 0].scatter(samples[:, 0], samples[:, 2], s=10, alpha=alpha)
axs[1, 0].arrow(0, 0, 3, 0, color='r', head_width=0.2, linewidth=2)
axs[1, 0].arrow(0, 0, 0, 3, color='b', head_width=0.2, linewidth=2)
axs[1, 0].text(3.5, 0, 'X', color='r', fontsize=12, weight='bold')
axs[1, 0].text(0, 3.5, 'Z', color='b', fontsize=12, weight='bold')
axs[1, 0].set_title('XZ Projection')
axs[1, 0].set_xlim(-4, 4)
axs[1, 0].set_ylim(-4, 4)
axs[1, 0].set_aspect('equal')

# 2D YZ Projection
axs[1, 1].scatter(samples[:, 1], samples[:, 2], s=10, alpha=alpha)
axs[1, 1].arrow(0, 0, 3, 0, color='g', head_width=0.2, linewidth=2)
axs[1, 1].arrow(0, 0, 0, 3, color='b', head_width=0.2, linewidth=2)
axs[1, 1].text(3.5, 0, 'Y', color='g', fontsize=12, weight='bold')
axs[1, 1].text(0, 3.5, 'Z', color='b', fontsize=12, weight='bold')
axs[1, 1].set_title('YZ Projection')
axs[1, 1].set_xlim(-4, 4)
axs[1, 1].set_ylim(-4, 4)
axs[1, 1].set_aspect('equal')

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

## Novel View Synthesis

Imagine this scenario:

- You're given a bunch of images of a scene
- You run Structure from Motion, and it works
- You'd like to know how the scene looks from a new point of view

How could you do this?

In [ ]:
# brainstorm here









### Key Question: scene representation

- 3D points ([example](https://www.cs.cornell.edu/~snavely/bundler/images/Colosseum.jpg))
  - you get these from structure from motion automatically
  - but they don't look real
- meshes
  - easy to render
  - very hard to compute
- patches ([example](https://www.di.ens.fr/cmvs/hall-all.jpg))
  - computable with effort
  - still looks kinda bad
- neural radiance field ([example](https://www.matthewtancik.com/nerf))
  - looks great!
  - takes days to compute
- blobs (today's topic)
  - easy to compute
  - easy to render
  - actually looks good
  - not as useful as a mesh
- Not going to talk about: signed distance functions (cool but niche)

## 3D Gaussian Splatting

We're done with the warmup. We're ready to learn about a cool paper that is making real headway on the novel view generation problem

Here's the key paper that introduced this idea: 

- [3D Gaussian Splatting for Real-Time Radiance Field Rendering](https://arxiv.org/abs/2308.04079)
- [[results page link](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/)]

## Demo: 2D Gaussian Splatting Optimization

The 3D version of Gaussian splatting requires a full 3D scene and known camera poses. But the core idea — **optimizing a set of Gaussians to reproduce an image** — works in 2D and is much easier to visualize. We'll start with a 2D live demo, and then talk about how to move that into 3D.

Here we'll:
1. Load a small target image
2. Initialize a set of random 2D Gaussians, each with learnable position, shape, color, and opacity
3. Render them onto a canvas using alpha compositing
4. Use gradient descent to minimize the difference between our rendering and the target

This is the same optimization loop that real 3DGS uses — just in 2D instead of 3D.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load and prepare target image
target_np = cv2.imread('../data/beans.jpg')
target_np = cv2.cvtColor(target_np, cv2.COLOR_BGR2RGB)
target_np = cv2.resize(target_np, (128, 128))
target = torch.tensor(target_np, dtype=torch.float32, device=device) / 255.0

H, W, _ = target.shape
print(f'Target image: {W}x{H}')
plt.imshow(target_np)
plt.title('Target image')
plt.axis('off')
plt.show()

In [ ]:
class L2Loss(nn.Module):
    """L2 (mean squared error) loss between rendered and target images."""
    
    def forward(self, rendered, target):
        return ((rendered - target) ** 2).mean()


class GaussianSplats2D(nn.Module):
    """A set of 2D Gaussians with learnable parameters."""
    
    def __init__(self, num_splats, H, W):
        super().__init__()
        # Positions: (num_splats, 2) in pixel coordinates
        self.means = nn.Parameter(torch.rand(num_splats, 2) * torch.tensor([W, H], dtype=torch.float32))
        # Log of scale (2 axes per Gaussian) - use log so scale is always positive
        self.log_scales = nn.Parameter(torch.log(torch.ones(num_splats, 2) * 8.0))
        # Rotation angle (radians)
        self.rotations = nn.Parameter(torch.rand(num_splats) * 2 * np.pi)
        # Color: (num_splats, 3) RGB - use sigmoid later to keep in [0,1]
        self.raw_colors = nn.Parameter(torch.randn(num_splats, 3))
        # Opacity: scalar per Gaussian - use sigmoid later to keep in [0,1]
        self.raw_opacities = nn.Parameter(torch.ones(num_splats) * 2.0)

    def render(self, H, W):
        # Build a coordinate grid: (H, W, 2)
        yy, xx = torch.meshgrid(torch.arange(H, device=self.means.device, dtype=torch.float32),
                                torch.arange(W, device=self.means.device, dtype=torch.float32),
                                indexing='ij')
        coords = torch.stack([xx, yy], dim=-1)  # (H, W, 2)
        
        # Get parameters
        scales = torch.exp(self.log_scales)  # (N, 2), always positive
        colors = torch.sigmoid(self.raw_colors)  # (N, 3), in [0,1]
        opacities = torch.sigmoid(self.raw_opacities)  # (N,), in [0,1]
        
        # Build 2x2 covariance matrices from scale and rotation
        cos_r = torch.cos(self.rotations)
        sin_r = torch.sin(self.rotations)
        # Rotation matrix R
        R = torch.stack([cos_r, -sin_r, sin_r, cos_r], dim=-1).reshape(-1, 2, 2)
        # Covariance = R @ diag(s^2) @ R^T
        S = torch.diag_embed(scales ** 2)  # (N, 2, 2)
        cov = R @ S @ R.transpose(-1, -2)  # (N, 2, 2)
        
        # Inverse covariance for the Gaussian exponent
        cov_inv = torch.linalg.inv(cov)  # (N, 2, 2)
        
        # Render by alpha compositing (back to front would be proper,
        # but for 2D we just accumulate; close enough for a demo)
        canvas = torch.zeros(H, W, 3, device=self.means.device)
        remaining_opacity = torch.ones(H, W, 1, device=self.means.device)
        
        for i in range(self.means.shape[0]):
            diff = coords - self.means[i]  # (H, W, 2)
            # Mahalanobis distance: diff @ cov_inv @ diff^T
            mahal = (diff @ cov_inv[i] * diff).sum(dim=-1)  # (H, W)
            alpha_map = opacities[i] * torch.exp(-0.5 * mahal)  # (H, W)
            alpha_map = alpha_map.unsqueeze(-1)  # (H, W, 1)
            
            # Front-to-back compositing
            canvas = canvas + remaining_opacity * alpha_map * colors[i]
            remaining_opacity = remaining_opacity * (1 - alpha_map)
        
        return canvas

In [ ]:
# Initialize the splats
num_splats = 500
model = GaussianSplats2D(num_splats, H, W).to(device)

# Render the initial (random) state
with torch.no_grad():
    init_render = model.render(H, W).cpu().numpy().clip(0, 1)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(target_np)
plt.title('Target')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(init_render)
plt.title(f'Initial rendering ({num_splats} Gaussians)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, clear_output

# Optimize!
optimizer = optim.Adam(model.parameters(), lr=0.015)
criterion = L2Loss()
num_iterations = 1000
snapshot_iters = {0, 10, 50, 100, 200, 500, 999}
snapshots = {}
losses = []
display_every = 10  # update the live plot every N iterations

# plot setup
fig, (ax_tgt, ax_render, ax_loss) = plt.subplots(1, 3, figsize=(15, 4))
ax_tgt.imshow(target_np)
ax_tgt.set_title('Target')
ax_tgt.axis('off')
render_im = ax_render.imshow(init_render)
ax_render.set_title('Iter 0')
ax_render.axis('off')
loss_line, = ax_loss.plot([], [])
ax_loss.set_xlabel('Iteration')
ax_loss.set_ylabel('L2 Loss')
ax_loss.set_title('Loss')
ax_loss.grid(True)
plt.tight_layout()

# training loop!
for iteration in range(num_iterations):
    optimizer.zero_grad()
    rendered = model.render(H, W)
    loss = criterion(rendered, target)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if iteration in snapshot_iters:
        snapshots[iteration] = rendered.detach().cpu().numpy().clip(0, 1)
    
    # plotting
    if iteration % display_every == 0 or iteration == num_iterations - 1:
        frame = rendered.detach().cpu().numpy().clip(0, 1)
        render_im.set_data(frame)
        ax_render.set_title(f'Iter {iteration} (loss={loss.item():.4f})')
        loss_line.set_data(range(len(losses)), losses)
        ax_loss.set_xlim(0, max(len(losses), 1))
        ax_loss.set_ylim(0, max(losses[0], losses[-1]) * 1.05)
        clear_output(wait=True)
        display(fig)

print('Done!')
plt.close(fig)

In [ ]:
# Show optimization progress
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('2D Gaussian Splatting: Optimization Progress', fontsize=14)

axes[0, 0].imshow(target_np)
axes[0, 0].set_title('Target')
axes[0, 0].axis('off')

for ax, it in zip(axes.flat[1:], sorted(snapshots.keys())):
    ax.imshow(snapshots[it])
    ax.set_title(f'Iter {it}')
    ax.axis('off')

plt.tight_layout()
plt.show()

### What just happened?

We optimized the **position, shape, color, and opacity** of each Gaussian to minimize the pixel-wise difference between our rendering and the target image.

3D Gaussian Splatting is conceptually similar, but there are some differences:
- The 3DGS Gaussians are 3D ellipsoids instead of 2D ellipses
- 3DGS projects through a camera model to get 2D splats on the image
- The loss is computed across many viewpoints, not just one
- Gaussians are added/removed during training (adaptive density control)
- Color depends on viewing direction (spherical harmonics) for specular effects


## Moving to 3D

![3DGS pipeline overview](../data/3dgs_pipeline.png)

*Figure 2 from Kerbl et al., "3D Gaussian Splatting for Real-Time Radiance Field Rendering", 2023.*

To make this work for real, do this:

- Given: a bunch of images of a scene
- Initialization: run SfM
  - yields poses for all cameras
  - create a small Gaussian for each SfM 3D point
  - Give each Gaussian parameters: $\mu$, $\Sigma$, color
- Refinement (training) loop:
  - Render the scene from the viewpoint of a real camera
  - Diff the rendering with the real image
  - Loss function: $l_1$ or $l_2$ sum over pixel diffs
  - Refine the position / shape / color of all the Gaussians with a gradient descent step
  - Possibly thin out (or add to) the number of Gaussians if they are too dense or too sparse somewhere

#### Color details:

- Easy mode: every Gaussian has an RGB color
- Hard mode: every Gaussian has a BRDF: *bidirectional radiance distribution function*. This is a graphics / rendering thing. Real materials look different from different angles. [[link: KB cloth rendering](https://www.cs.cornell.edu/projects/ctcloth/)]
  - Use spherical harmonics. (Think Taylor series, but in 3D for things that are kind of spherish)

#### Blending details:

To do this right, 

- Each Gaussian has an alpha opacity
- The order of blending matters!
- Sort the splats by depth. Blend back to front.

#### If there's extra time:

Other fun details:

- Splatting is a hack for volume rendering
- The choice of loss function matters. The authors use a sum of $L_1$ and [D-SSIM](https://github.com/kornelski/dssim?tab=readme-ov-file). D-SSIM is a tweak on [SSIM](https://en.wikipedia.org/wiki/Structural_similarity_index_measure)
- Point cloud to 3D mesh approaches weren't that good for SfM points, but this [3DGS-to-mesh method](https://github.com/Anttwo/SuGaR) seems to really work!